In [0]:
# ============================================================
# Cell 1 - Install dependencies
# ============================================================
%pip install requests
dbutils.library.restartPython()

In [0]:
# databricks secrets create-scope media
# databricks secrets put-secret media tmdb_api_key

In [0]:
# ============================================================
# Cell 2 - Configuration
# ============================================================
import requests
import time
import json
from datetime import datetime
from pyspark.sql import Row
import pyspark.sql.functions as F

TMDB_API_KEY    = dbutils.secrets.get(scope="media", key="tmdb_api_key")
BASE_URL        = "https://api.themoviedb.org/3"
CATALOG         = "media"
SCHEMA          = "bronze_tmdb"

# TMDB network IDs for major streaming platforms
STREAMING_NETWORKS = {
    213:  "Netflix",
    2739: "Disney+",
    1024: "Amazon",
    2552: "Apple TV+",
}

print("Configuration loaded!")

In [0]:
# Cell 3 - Create schema
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
# ============================================================
# Cell 4 - Reusable paginated fetch helper
# ============================================================
def fetch_pages(endpoint: str, params: dict, max_pages: int = 10) -> list[dict]:
    """
    Fetches up to max_pages from a TMDB paginated endpoint.
    Returns a list of dicts with raw_payload + ingestion metadata.
    """
    results     = []
    ingested_at = datetime.utcnow().isoformat()
    params      = {**params, "api_key": TMDB_API_KEY, "page": 1}

    for page in range(1, max_pages + 1):
        params["page"] = page
        resp = requests.get(f"{BASE_URL}{endpoint}", params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()

        for item in data.get("results", []):
            results.append({
                "raw_payload": json.dumps(item),
                "source_page": page,
                "endpoint":    endpoint,
                "ingested_at": ingested_at,
            })

        if page >= data.get("total_pages", 1):
            break

        time.sleep(0.25)  # stay within TMDB rate limits

    print(f"  Fetched {len(results):,} records from {endpoint}")
    return results

In [0]:
# ============================================================
# Cell 5 - Ingest movies
# ============================================================
movie_records = []

# Popular movies — what's trending now
movie_records += fetch_pages("/discover/movie", {
    "sort_by":        "popularity.desc",
    "vote_count.gte": 100,
}, max_pages=20)

# Top rated movies — historical quality signal
movie_records += fetch_pages("/discover/movie", {
    "sort_by":        "vote_average.desc",
    "vote_count.gte": 1000,
}, max_pages=20)

df_movies = (spark
    .createDataFrame([Row(**r) for r in movie_records])
    .withColumn("media_type", F.lit("movie")))

(df_movies.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.raw_movies"))

print(f"raw_movies: {df_movies.count():,} rows saved")

In [0]:
# ============================================================
# Cell 6 - Ingest TV shows
# ============================================================
tv_records = []

# Popular TV shows
tv_records += fetch_pages("/discover/tv", {
    "sort_by":        "popularity.desc",
    "vote_count.gte": 50,
}, max_pages=20)

# TV shows by streaming network
for network_id, network_name in STREAMING_NETWORKS.items():
    print(f"  Fetching {network_name}...")
    tv_records += fetch_pages("/discover/tv", {
        "with_networks": network_id,
        "sort_by":       "popularity.desc",
    }, max_pages=10)

df_tv = (spark
    .createDataFrame([Row(**r) for r in tv_records])
    .withColumn("media_type", F.lit("tv")))

(df_tv.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.raw_tv_shows"))

print(f"raw_tv_shows: {df_tv.count():,} rows saved")

In [0]:
# ============================================================
# Cell 7 - Ingest genre reference data
# ============================================================
for media in ["movie", "tv"]:
    resp   = requests.get(
        f"{BASE_URL}/genre/{media}/list",
        params={"api_key": TMDB_API_KEY}
    )
    genres = resp.json()["genres"]

    df_genres = (spark
        .createDataFrame(genres)
        .withColumn("media_type", F.lit(media)))

    (df_genres.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{SCHEMA}.raw_genres_{media}"))

    print(f"raw_genres_{media}: {len(genres)} rows saved")

In [0]:
# ============================================================
# Cell 8 - Sanity check
# ============================================================
tables = [
    "raw_movies",
    "raw_tv_shows",
    "raw_genres_movie",
    "raw_genres_tv",
]

print("=== Bronze Layer Summary ===")
for table in tables:
    count = spark.table(f"{CATALOG}.{SCHEMA}.{table}").count()
    print(f"  {CATALOG}.{SCHEMA}.{table}: {count:,} rows")

print("\n=== Sample raw_payload (movie) ===")
sample = spark.table(f"{CATALOG}.{SCHEMA}.raw_movies").limit(1).collect()[0]
print(json.dumps(json.loads(sample["raw_payload"]), indent=2))